# Exercise 03 — Perplexity

**Perplexity** is the standard way to measure how good a language model is: how *surprised* the model is by text it has not seen before. Lower is better.

In this notebook you build a **bigram language model** by counting, turn those counts into probabilities, and use them to compute perplexity — all by hand on a corpus small enough to check every number yourself.

## What you will do
1. **Count** unigrams and bigrams in a tiny training corpus.
2. **Estimate probabilities** from the counts, first without smoothing, then with **add-1 (Laplace) smoothing**.
3. **Compute perplexity** of a test sentence.
4. **Compare sentences** and see what perplexity actually tells you.
5. **Multiple-choice questions** — to check your understanding.

## How to work through it
- Run the cells **in order** and fill in **every `# TODO`**.
- Tasks are numbered (**1.1**, **1.2**, …). Tasks that say *Your answer here* want a short written answer, not code.
- The **✅ Check** cells verify your work automatically — make sure they pass before moving on.

> No downloads and no GPU needed. The whole notebook runs in a second.


In [ ]:
import math
from collections import defaultdict
from typing import List


## 1. The data and the counts

We use a **tiny training corpus** of four sentences. Being able to verify every count by hand is the whole point: everything in this notebook could be done on a corpus of a billion sentences, but then you could not check it.

The `test_sentence` is what we will measure perplexity on. Note that it never appears in the training corpus, although all of its words do.


In [ ]:
# Training corpus
training_corpus = [
    "the cat sat on the mat",
    "the dog ran in the park",
    "a cat likes fish",
    "the mat is soft",
]

# The sentence we will score
test_sentence = "the cat likes the mat"


### Sentence boundaries

Before counting, we wrap every sentence in two special tokens:

- `<s>` marks the **start** of a sentence,
- `</s>` marks the **end**.

They matter because they let the model learn which words *begin* a sentence — `P(the | <s>)` — and which ones *end* it. Without `</s>` the model would have no way to assign probability to a sentence stopping.

**1.1 Implement `tokenize_with_boundaries`: split a sentence on whitespace and add `<s>` at the front and `</s>` at the end.**

Hint: `sentence.split()` gives you the words; you only need to concatenate three lists.


In [ ]:
def tokenize_with_boundaries(sentence: str) -> List[str]:
    """Split into words and wrap in sentence-boundary tokens."""
    return ["<s>"] + sentence.split() + ["</s>"]


In [ ]:
# ✅ Check your tokenizer
assert tokenize_with_boundaries("the mat is soft") == ["<s>", "the", "mat", "is", "soft", "</s>"]
assert tokenize_with_boundaries("hi") == ["<s>", "hi", "</s>"]
print("Looks good ✅")


### Counting unigrams and bigrams

A **bigram model** approximates the probability of a sentence as a product of conditional probabilities, where each word depends only on the one before it:

\begin{align}
P(w_1, \dots, w_n) \approx \prod_{i=1}^{n} P(w_i \mid w_{i-1})
\end{align}

To estimate those probabilities we need two sets of counts:

- **unigram counts** — how often each word occurs, $C(w)$,
- **bigram counts** — how often each *pair of adjacent* words occurs, $C(w_{i-1}, w_i)$.

**1.2 Fill in the two counting loops.**

Hints:
- Both dictionaries are `defaultdict(int)`, so `counts[key] += 1` works even for a key you have not seen before.
- A bigram is the **tuple** `(tokens[i], tokens[i + 1])` — use a tuple, not a string, as the key.
- The bigram loop stops at `len(tokens) - 1`, because the last token has no successor.


In [ ]:
bigram_counts = defaultdict(int)
unigram_counts = defaultdict(int)

print("Tokenized corpus:")
for sentence in training_corpus:
    tokens = tokenize_with_boundaries(sentence)
    print(f"   {tokens}")

    # Count the unigrams
    for token in tokens:
        unigram_counts[token] += 1

    # Count the bigrams
    for i in range(len(tokens) - 1):
        bigram_counts[(tokens[i], tokens[i + 1])] += 1

# Freeze the counts as plain dicts, so looking up an unknown word
# can no longer silently insert it with a count of 0.
unigram_counts = dict(unigram_counts)
bigram_counts = dict(bigram_counts)

print("\nUnigram counts:")
for word, count in sorted(unigram_counts.items()):
    print(f"   {word:<6} {count}")

print("\nBigram counts:")
for bigram, count in sorted(bigram_counts.items()):
    print(f"   {str(bigram):<20} {count}")

test_tokens = tokenize_with_boundaries(test_sentence)
print(f"\nTokenized test sentence: {test_tokens}")


In [ ]:
# ✅ Check your counts — verify a few of them by hand against the corpus above
assert unigram_counts["the"] == 5, f"'the' occurs 5 times, you counted {unigram_counts.get('the')}"
assert unigram_counts["<s>"] == 4 and unigram_counts["</s>"] == 4, "one <s> and one </s> per sentence"
assert unigram_counts["cat"] == 2
assert bigram_counts[("the", "mat")] == 2, "'the mat' occurs in sentence 1 and sentence 4"
assert bigram_counts[("<s>", "the")] == 3, "three sentences start with 'the'"
assert ("likes", "the") not in bigram_counts, "'likes the' never occurs in training"

vocab_size = len(unigram_counts)
print(f"Vocabulary size (V): {vocab_size}")
print(f"Distinct bigrams   : {len(bigram_counts)}")
print("Looks good ✅")


## 2. From counts to probabilities

### The maximum-likelihood estimate

The obvious way to turn counts into a conditional probability is to divide:

\begin{align}
P_{\text{MLE}}(w_i \mid w_{i-1}) = \frac{C(w_{i-1}, w_i)}{C(w_{i-1})}
\end{align}

**2.1 Implement this unsmoothed estimate.**

Hint: use `.get(key, 0)` so that a bigram the model has never seen returns `0` instead of raising a `KeyError`.


In [ ]:
def get_bigram_prob_mle(w1: str, w2: str) -> float:
    """Unsmoothed maximum-likelihood estimate of P(w2 | w1)."""
    return bigram_counts.get((w1, w2), 0) / unigram_counts.get(w1, 0)


for w1, w2 in [("the", "cat"), ("cat", "likes"), ("likes", "the")]:
    print(f"P({w2:<5} | {w1:<5}) = {get_bigram_prob_mle(w1, w2):.4f}")


**2.2 One of those three probabilities is exactly zero.** The probability of a whole sentence is the *product* of its bigram probabilities, and perplexity is computed from $\log_2$ of those probabilities.

What happens to the sentence probability when one bigram gets probability 0? And what happens when you take $\log_2$ of it?

---

*Answer:*

The whole sentence gets probability **0**. It is a product, and a single zero factor wipes out everything else — no matter how likely the other bigrams are. The culprit here is `P(the | likes)`: *likes the* never occurs in the training corpus, even though it is a perfectly normal phrase.

Log space does not rescue it: $\log_2 0 = -\infty$ (Python's `math.log2(0)` actually raises `ValueError: math domain error`), so the average log-probability is $-\infty$ and the perplexity $2^{+\infty}$ is **infinite**. One unseen bigram makes the model "infinitely surprised", and any test sentence containing one can no longer be scored or compared. The MLE gives *all* the probability mass to what it happened to see in training — which is exactly the problem smoothing fixes.

---


### Add-1 (Laplace) smoothing

The fix is to pretend we saw every possible bigram **one extra time**. We add 1 to every bigram count, and — to keep the result a valid probability distribution — add the vocabulary size $V$ to every denominator:

\begin{align}
P_{\text{add-1}}(w_i \mid w_{i-1}) = \frac{C(w_{i-1}, w_i) + 1}{C(w_{i-1}) + V}
\end{align}

Now no bigram ever gets probability 0, so the model can assign a non-zero probability to any sentence over its vocabulary.

**2.3 Implement the smoothed estimate.**

Hint: `vocab_size` was computed for you in the check cell above. Use `.get(key, 0)` again for both counts.


In [ ]:
def get_bigram_prob_smoothed(w1: str, w2: str) -> float:
    """P(w2 | w1) with add-1 (Laplace) smoothing."""
    numerator = bigram_counts.get((w1, w2), 0) + 1
    denominator = unigram_counts.get(w1, 0) + vocab_size
    return numerator / denominator


print(f"unsmoothed  P(the | likes) = {get_bigram_prob_mle('likes', 'the'):.4f}")
print(f"smoothed    P(the | likes) = {get_bigram_prob_smoothed('likes', 'the'):.4f}")


In [ ]:
# ✅ Check your smoothing
# For a fixed first word, the probabilities over the whole vocabulary must sum to 1.
for w1 in ["the", "cat", "<s>"]:
    total = sum(get_bigram_prob_smoothed(w1, w2) for w2 in unigram_counts)
    print(f"sum over w2 of P(w2 | {w1:<4}) = {total:.10f}")
    assert math.isclose(total, 1.0, abs_tol=1e-9), "a distribution has to sum to 1 — check your denominator"

# Nothing may be zero any more
assert get_bigram_prob_smoothed("likes", "the") > 0, "smoothing should remove all zeros"
print("Looks good ✅")


**2.4 Print the probability of every bigram in the test sentence, and accumulate their log-probabilities.**

We work in $\log_2$ rather than with the raw probabilities because multiplying many small numbers underflows to 0 very quickly. Sums of logs are numerically safe — and, as you will see next, exactly what the perplexity formula needs.

Hints:
- `math.log2(prob)` gives the base-2 logarithm.
- The log-probabilities are all **negative**, since every probability is below 1.


In [ ]:
log_prob_sum = 0.0
n_bigrams = len(test_tokens) - 1

print(f"Bigram probabilities for: {test_sentence!r}\n")
for i in range(n_bigrams):
    w1, w2 = test_tokens[i], test_tokens[i + 1]

    prob = get_bigram_prob_smoothed(w1, w2)
    log_prob = math.log2(prob)
    log_prob_sum += log_prob

    c_bi = bigram_counts.get((w1, w2), 0)
    c_uni = unigram_counts.get(w1, 0)
    print(f"   P({w2:<5} | {w1:<5}) = ({c_bi} + 1) / ({c_uni} + {vocab_size}) = {prob:.4f}"
          f"   log2 = {log_prob:+.4f}")

print(f"\nSum of log2 probabilities: {log_prob_sum:.4f}   over N = {n_bigrams} bigrams")


## 3. Perplexity

Perplexity is the average log-probability, negated and exponentiated back out of log space:

\begin{align}
\text{Perplexity} = 2^{-\frac{1}{N} \sum_{i=1}^{N} \log_2 P(w_i \mid w_{i-1})}
\end{align}

where $N$ is the number of bigrams scored. You can read it as **"on average, how many words is the model choosing between at each step?"** A perplexity of 10 means the model is about as uncertain as if it were picking uniformly among 10 words. **Lower is better.**

**3.1 Compute the perplexity of the test sentence from `log_prob_sum` and `n_bigrams`.**

Hint: in Python, `2 ** x` raises 2 to the power of `x`.


In [ ]:
avg_log_prob = log_prob_sum / n_bigrams
perplexity = 2 ** (-avg_log_prob)

print(f"Sum of log probabilities: {log_prob_sum:.4f}")
print(f"Number of bigrams (N)   : {n_bigrams}")
print(f"Average log probability : {avg_log_prob:.4f}")
print(f"Perplexity              : {perplexity:.4f}")


In [ ]:
# ✅ Check your perplexity
assert 1 <= perplexity <= vocab_size + 1, "perplexity should sit between 1 and roughly the vocabulary size"
assert math.isclose(perplexity, 2 ** (-log_prob_sum / n_bigrams), rel_tol=1e-9)
print(f"A perplexity of {perplexity:.1f} on a vocabulary of {vocab_size} words.")
print("Looks good ✅")


### Scoring any sentence

**3.2 Wrap the whole calculation into one reusable function**, so we can compare sentences against each other.


In [ ]:
def sentence_perplexity(sentence: str) -> float:
    """Perplexity of `sentence` under the smoothed bigram model."""
    tokens = tokenize_with_boundaries(sentence)
    n = len(tokens) - 1

    log_prob_sum = sum(
        math.log2(get_bigram_prob_smoothed(tokens[i], tokens[i + 1]))
        for i in range(n)
    )

    return 2 ** (-log_prob_sum / n)


# Should agree with what you computed above
print(f"{sentence_perplexity(test_sentence):.4f}  vs  {perplexity:.4f}")
assert math.isclose(sentence_perplexity(test_sentence), perplexity, rel_tol=1e-9)
print("Looks good ✅")


**3.3 Score the sentences below and add one of your own.** They range from "copied straight out of the training data" to "the model has never seen any of these words".


In [ ]:
sentences = [
    "the cat sat on the mat",        # verbatim from the training corpus
    "the cat likes the mat",         # familiar words, new combination
    "the dog ran in the park",       # also verbatim
    "a soft cat likes the park",     # all words known, 3 of 7 bigrams new
    "quantum physics is difficult",  # almost nothing is known
    "the dog sat on the soft mat",   # your own: known words, a mix of seen and new bigrams
]

for s in sentences:
    print(f"{sentence_perplexity(s):8.2f}   {s}")


**3.4 Explain the ordering.** Why do the sentences taken verbatim from the training corpus score lowest, and the last one highest? What does the number for `"quantum physics is difficult"` tell you about how much this model really knows — and would you trust perplexity to compare two models trained on *different* vocabularies?

---

*Answer:*

| Perplexity | Sentence | Why |
|---|---|---|
| 8.03, 8.37 | the two verbatim training sentences | every bigram was seen in training |
| 8.93 | the cat likes the mat | one unseen bigram (*likes the*) |
| 12.17 | a soft cat likes the park | 3 of its 7 bigrams (*a soft*, *soft cat*, *likes the*) never occurred |
| 16.93 | quantum physics is difficult | 3 of its 4 words are not in the vocabulary at all |

The ranking simply tracks how much of each sentence the model saw during training: seen bigrams get large numerators, unseen ones only the $+1$. Note that even the verbatim sentences are nowhere near a perplexity of 1 — with only four training sentences, add-1 smoothing (adding $V = 16$ to every denominator) hands most of the probability mass to bigrams that never occurred.

**"quantum physics is difficult".** For a history word the model has never seen, $C(w_{i-1}) = 0$ and the smoothed estimate collapses to $\frac{0 + 1}{0 + V} = \frac{1}{16}$ — a uniform guess. A perplexity of ≈ 17, about the vocabulary size, means the model does no better than picking uniformly at random among its words: it knows **nothing** about this sentence. Strictly, even that is flattering — *quantum*, *physics* and *difficult* are not in the vocabulary, so the model should not be able to give them any probability. Real models map such words to a special `<UNK>` token.

**Different vocabularies? No.** Uniform guessing scores exactly $V$, so perplexity scales with the size of the vocabulary. A model with a smaller vocabulary, or one that maps many rare words to `<UNK>`, gets a lower perplexity without being any better — it just has fewer outcomes to choose between. The same goes for different tokenizers (words vs. sub-words), where the per-token average is taken over different units. Perplexities are only comparable on the **same test set with the same vocabulary / tokenizer**.

---


## 4. MCQ

Answer each question by writing the letter of your choice (A–D) after **Answer:**.

---

### 4.1. Definition of Perplexity

What does perplexity measure in language models?

A. The total number of words in the test set<br>
B. The unpredictability or "surprise" of a model when predicting text<br>
C. The size of the vocabulary<br>
D. The average frequency of bigrams<br>

**Answer:** B ✅

---

### 4.2. Formula for Perplexity

Which of the following is the correct formula for perplexity of a test set with *N* tokens?

A. $\text{Perplexity} = \frac{1}{N} \sum \log P(w_i)$<br>
B. $\text{Perplexity} = 2^{-\frac{1}{N} \sum \log_2 P(w_i \mid context)}$<br>
C. $\text{Perplexity} = \prod_{i=1}^{N} P(w_i)$<br>
D. $\text{Perplexity} = N^{\sum P(w_i)}$<br>

**Answer:** B ✅

---

### 4.3. Interpretation of Perplexity

If a model has **lower perplexity** on a test set, what does it mean?

A. The model is more confident and better at predicting the test data<br>
B. The model is overfitting<br>
C. The vocabulary size is smaller<br>
D. The training corpus is too simple<br>

**Answer:** A ✅

---

### 4.4. Smoothing and Perplexity

Why is **add-1 smoothing (Laplace smoothing)** used when computing perplexity?

A. To reduce the vocabulary size<br>
B. To ensure unseen word pairs do not get zero probability<br>
C. To increase the average log probability<br>
D. To make the model faster to train<br>

**Answer:** B ✅

---

### 4.5. Why work in log space

Why do we sum $\log_2$ probabilities instead of multiplying the probabilities directly?

A. Because the logarithm makes the model more accurate<br>
B. Because a product of many small probabilities underflows to zero in floating point<br>
C. Because perplexity is only defined for logarithms<br>
D. Because it removes the need for smoothing<br>

**Answer:** B ✅

---

### 4.6. Comparing models

Two language models report perplexities of 45 and 120 on the same test set with the same vocabulary. Which statement is correct?

A. The model with perplexity 120 predicts the test set better<br>
B. The model with perplexity 45 predicts the test set better<br>
C. The two numbers cannot be compared<br>
D. Perplexity 45 means the model is 45% accurate<br>

**Answer:** B ✅
